# Qwen2-VL 4-bit QLoRA Fine-Tuning Guide for Colab

## 1. Where to find the model?
The model is completely free and open-source. It is hosted on **Hugging Face**, which is the central hub for AI models.
- **Model Link:** [Qwen/Qwen2-VL-7B-Instruct](https://huggingface.co/Qwen/Qwen2-VL-7B-Instruct)
- You do not need to download it manually. The `transformers` library will download it directly into your Colab environment using the model ID: `"Qwen/Qwen2-VL-7B-Instruct"`.

---
## 2. Environment Setup
First, we need to install the libraries required to load the model and compress it to 4-bit.

In [ ]:
!pip install -q git+https://github.com/huggingface/transformers
!pip install -q accelerate bitsandbytes peft qwen-vl-utils

## 3. Loading the Model in 4-bit Precision
The 7B model normally requires ~14GB of VRAM just to load, and fine-tuning takes even more. By using `BitsAndBytesConfig`, we quantize (compress) the model down to 4-bit precision, making it take only ~4-5GB of VRAM.

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

model_id = "Qwen/Qwen2-VL-7B-Instruct"

# 1. Define the 4-bit quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16, # Uses bfloat16 for computation to maintain accuracy
    bnb_4bit_use_double_quant=True,        # Saves even more memory
    bnb_4bit_quant_type="nf4"              # Normalized Float 4 (best for weights)
)

# 2. Load the Processor (Handles image and text processing)
processor = AutoProcessor.from_pretrained(model_id)

# 3. Load the actual model quantized to 4-bit
print("Downloading and loading the model ...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",               # Automatically puts the model on the T4 GPU
    quantization_config=quantization_config
)
print("Model loaded successfully in 4-bit!")

## 4. Applying LoRA (Low-Rank Adaptation)
You cannot fine-tune a 4-bit model directly. Instead, we use **LoRA** (via the `peft` library). LoRA freezes the original massive 7B model and attaches tiny "adapter" layers on top. 
You only train these tiny adapters, which requires very little memory and is extremely fast.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. Prepare the 4-bit model for training (sets up gradients properly)
model = prepare_model_for_kbit_training(model)

# 2. Define the LoRA configuration
lora_config = LoraConfig(
    r=16,                             # The rank of the adapter (16 is a good balance of speed/accuracy)
    lora_alpha=32,                    # Scaling factor
    target_modules=["q_proj", "v_proj"], # Which layers of the AI to attach the adapters to
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 3. Wrap the base model with the LoRA adapters
model = get_peft_model(model, lora_config)

# Print how many parameters you are actually training
model.print_trainable_parameters()

## 5. Next Steps for 7-12 Extracts
From here, the model is ready to be trained. You will need to:
1. Convert your 7-12 images and their corresponding JSON targets into a Hugging Face `Dataset`.
2. Pass them through the `processor`.
3. Use the `SFTTrainer` (Supervised Fine-Tuning Trainer) from the `trl` library to start the training loop.